# Follow-the-DROW — Model Training Notebook

This notebook trains all **five learnable detectors** on the FROG dataset and evaluates
them alongside the **rule-based AlgorithmicDetector**.  Each model section produces:

1. **Loss curve** — training and validation loss per epoch  
2. **AUC curve** — per-class validation AUC at regular intervals  

A final **comparison cell** evaluates every model on the held-out **test split** and
renders a side-by-side bar chart.

> **Runtime:** select *Runtime → Change runtime type → T4 GPU* for best performance.

---

| # | Model | Input | GPU | Notes |
|---|-------|-------|-----|-------|
| 0 | AlgorithmicDetector | scan | — | rule-based, no training |
| 1 | DrowDetector | cutout | ✓ | original DROW WNet3xLF2p |
| 2 | DrSpaamDetector | cutout | ✓ | SpatialAttention + TemporalAttention |
| 3 | FullScanCNNDetector | full scan | ✓ | dilated CNN + GRU |
| 4 | SpaceTimeCNNDetector | full scan | ✓ | 2-D space-time convolution |
| 5 | FullScanTransformerDetector | full scan | ✓ | dilated CNN + beam attn + GRU |

> **Local Windows users (DirectML):** `FullScanCNN` and `FullScanTransformer` use
> `nn.GRU`, which is not supported on the DirectML backend.  Add `force_cpu=True`
> to their `_default_args(...)` calls when running locally with DirectML.

In [ ]:
# @title Setup — run once before any other cell
# Fill in REPO_URL, then run.  Skip entirely if running locally.

REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'  # ← edit this
BRANCH   = 'master'

import subprocess, os, sys
from pathlib import Path

# ── Option A: clone from GitHub (default) ──────────────────────────────────
CLONE_DIR = Path('/content/follow_the_drow')
if not CLONE_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True,
    )
os.chdir(str(CLONE_DIR / 'utils'))

# ── Option B: load from Google Drive ───────────────────────────────────────
# Uncomment the block below (and comment out Option A) if you uploaded the
# repo to Drive instead of cloning from GitHub.
#
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/follow_the_drow'   # ← edit
# os.chdir(f'{DRIVE_PATH}/utils')

# ── Install the library ─────────────────────────────────────────────────────
# --no-deps avoids version conflicts with Colab's pre-installed torch/numpy.
# The C++ extension (AlgorithmicDetector binding) is compiled automatically
# by pybind11 via setuptools — no extra tools needed.
# FROG data is NOT downloaded here; it is fetched automatically (~few hundred
# MB per split) on the first FROG_Dataset() call in the notebook.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '../library'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'certifi', 'tqdm', 'scikit-learn'],
    check=True,
)

print(f'Working directory : {Path.cwd()}')
print('Setup complete.')

In [ ]:
import sys
import math
import warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})
plt.style.use('seaborn-v0_8-whitegrid')

# After the setup cell runs %cd / os.chdir, Path.cwd() is already utils/
_nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))

from train import (
    train_model, _default_args,
    _setup_datasets, _build_model, load_checkpoint,
    evaluate_auc, detect_device,
)

device, using_dml = detect_device()

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}  (DirectML={using_dml})')
print(f'Notebook : {_nb_dir}')

In [ ]:
_PALETTE = plt.cm.tab10.colors


# ---------------------------------------------------------------------------
# Training history plot
# ---------------------------------------------------------------------------

def plot_history(history: dict, model_name: str):
    """Plot loss and validation AUC curves from a train_model() history dict."""
    epochs     = history['epochs']
    auc_epochs = history['auc_epochs']
    stopped    = history.get('stopped_epoch', epochs[-1] if epochs else 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    title = model_name
    if epochs and stopped < epochs[-1]:
        title += f'  (early stop @ epoch {stopped})'
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Loss ---
    ax1.plot(epochs, history['train_loss'], label='train', lw=2)
    if history['val_loss']:
        ax1.plot(epochs, history['val_loss'], label='val', lw=2, ls='--')
    if epochs and stopped < epochs[-1]:
        ax1.axvline(stopped, color='red', ls=':', lw=1.2, label=f'stop @ {stopped}')
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss')
    ax1.legend()

    # AUC ---
    if auc_epochs:
        ax2.plot(auc_epochs, history['val_auc_agnostic'],
                 label='agnostic', lw=2.2, color='black')
        ax2.plot(auc_epochs, history['val_auc_wc'],
                 label='wc (wheelchair)', ls='--', lw=1.5)
        ax2.plot(auc_epochs, history['val_auc_wa'],
                 label='wa (walker)',     ls='--', lw=1.5)
        ax2.plot(auc_epochs, history['val_auc_wp'],
                 label='wp (person)',     ls='--', lw=1.5)
        ax2.set_ylim(0, 1)
        final = history['val_auc_agnostic'][-1]
        ax2.set(xlabel='Epoch', ylabel='AUC',
                title=f'Validation AUC  (final agnostic = {final:.1%})')
        ax2.legend(fontsize=9)
    else:
        ax2.text(0.5, 0.5, 'No AUC computed\n(set auc_every > 0)',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=11)
        ax2.set_title('Validation AUC')

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# Neural model evaluation
# ---------------------------------------------------------------------------

def eval_neural(detector_name: str, ckpt_path, split: str = 'test',
                batch_size: int = 4) -> dict | None:
    """Load a checkpoint and compute AUC on *split*."""
    ckpt = Path(ckpt_path)
    if not ckpt.exists():
        print(f'  [{detector_name}] checkpoint not found: {ckpt}')
        return None
    force_cpu = detector_name in ('fullscan_cnn', 'fullscan_transformer')
    dev = 'cpu' if force_cpu else device
    args = _default_args(detector=detector_name, dataset='frog',
                         train_split='train', val_split=split)
    _, test_ds, cfg = _setup_datasets(args)
    net = _build_model(_default_args(detector=detector_name)).to(dev)
    load_checkpoint(ckpt, net)
    return evaluate_auc(net, test_ds, cfg, device=dev, batch_size=batch_size)


# ---------------------------------------------------------------------------
# Algorithmic detector evaluation (precision / recall / F1)
# ---------------------------------------------------------------------------

def eval_algorithmic(split: str = 'test', eval_r: float = 0.5) -> dict:
    """
    Run AlgorithmicDetector on *split* and compute precision, recall, F1.

    The detector produces (x, y) positions without confidence scores, so
    AUC cannot be computed.  We match each prediction to GT within eval_r
    metres and report binary TP/FP/FN statistics.
    """
    from follow_the_drow.detectors import AlgorithmicDetector
    from follow_the_drow.datasets import FROG_Dataset
    from tqdm.auto import tqdm

    ds = FROG_Dataset(split=split)
    tp = fp = fn = 0

    for seq in tqdm(range(len(ds.det_id)), desc='AlgorithmicDetector eval', leave=False):
        algo = AlgorithmicDetector(verbose=False)   # fresh instance per sequence (stateful)
        for det_idx in range(len(ds.det_id[seq])):
            iscan = ds.idet2iscan[seq][det_idx]
            scans_hist, odoms_hist = ds.get_scan(seq, iscan, ds.time_frame)
            raw   = algo.forward_one(scans_hist[-1], odoms_hist[-1]['xya'])
            preds = np.array(raw).reshape(-1, 2) if len(raw) else np.zeros((0, 2))

            gt_rp = (ds.det_wc[seq][det_idx]
                     + ds.det_wa[seq][det_idx]
                     + ds.det_wp[seq][det_idx])
            gt_xy = np.array([(-r * np.sin(p), r * np.cos(p)) for r, p in gt_rp]
                             ).reshape(-1, 2)

            matched_gt = set()
            for px, py in preds:
                if len(gt_xy):
                    dists = np.hypot(gt_xy[:, 0] - px, gt_xy[:, 1] - py)
                    best  = int(np.argmin(dists))
                    if dists[best] < eval_r and best not in matched_gt:
                        tp += 1
                        matched_gt.add(best)
                        continue
                fp += 1
            fn += max(0, len(gt_xy) - len(matched_gt))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return dict(precision=precision, recall=recall, f1=f1)


# ---------------------------------------------------------------------------
# Final comparison chart
# ---------------------------------------------------------------------------

def plot_comparison(neural_results: dict, algo: dict | None = None):
    """
    Grouped bar chart — per-class AUC for neural models, F1 for algorithmic.

    neural_results : {label: aucs_dict}  aucs_dict has keys agnostic/wc/wa/wp
    algo           : {precision, recall, f1} or None
    """
    classes   = ['agnostic', 'wc', 'wa', 'wp']
    cls_label = ['Agnostic', 'WC', 'WA', 'WP']
    n_models  = len(neural_results) + (1 if algo else 0)
    x         = np.arange(n_models)
    width     = 0.18

    fig, ax = plt.subplots(figsize=(max(10, n_models * 2), 5))

    for ci, (cls, lbl) in enumerate(zip(classes, cls_label)):
        vals = [r[cls] if r else 0.0 for r in neural_results.values()]
        if algo:
            vals.append(algo['f1'] if cls == 'agnostic' else float('nan'))
        offset = (ci - 1.5) * width
        bars   = ax.bar(x + offset, vals, width, label=lbl,
                        color=_PALETTE[ci], alpha=0.85)
        for bar, v in zip(bars, vals):
            if math.isnan(v):
                continue
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.012,
                    f'{v:.0%}', ha='center', va='bottom', fontsize=7.5)

    labels = list(neural_results.keys())
    if algo:
        labels.append(f'Algorithmic\n(F1={algo["f1"]:.1%})')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('AUC  /  F1')
    ax.set_title('Test-set comparison — all models', fontsize=13, fontweight='bold')
    ax.legend(title='Class', loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


print('Helpers defined.')

## Model 0 — AlgorithmicDetector

A **rule-based** detector using laser scan clustering and a simple leg/chest geometry model.
No neural network, no training required.

**Algorithm overview:**
1. Segment beams into clusters separated by range discontinuities.
2. Match clusters geometrically to person/leg/chest size constraints.
3. Track candidates across frames using frequency and uncertainty.

Because the detector outputs discrete (x, y) positions with no confidence score,
precision-recall **AUC is not applicable**.  We report **precision / recall / F1**
at a fixed matching radius of 0.5 m instead.

> This cell may take a few minutes on the full test split.

In [ ]:
algo_result = eval_algorithmic(split='test', eval_r=0.5)

print(f"AlgorithmicDetector — test split")
print(f"  Precision : {algo_result['precision']:.1%}")
print(f"  Recall    : {algo_result['recall']:.1%}")
print(f"  F1        : {algo_result['f1']:.1%}")

## Model 1 — DrowDetector

Original **DROW WNet3xLF2p** architecture from the DROW paper.

- **Input:** per-beam polar cutout windows  `(N_beams, T=5, 48)`
- **Architecture:** three parallel 1-D dilated convolution streams + late fusion
- **DirectML:** ✓ compatible (~12× speedup over CPU in eval)

In [ ]:
args_drow = _default_args(
    detector    = 'drow',
    dataset     = 'frog',
    epochs      = 30,
    lr          = 1e-3,
    dropout     = 0.5,
    patience    = 5,
    lr_schedule = 'plateau',
    auc_every   = 5,
    out         = _nb_dir / 'weights_drow_frog.pth',
)

history_drow = train_model(args_drow)
plot_history(history_drow, 'DrowDetector')

## Model 2 — DrSpaamDetector

**DR-SPAAM** (Distance-Robust Spatial Attention and Auto-regressive Model).

- **Input:** per-beam polar cutout windows  `(N_beams, T=5, 48)`
- **Architecture:** auto-regressive `SpatialAttention` (local ±3-beam window) + conv-based `TemporalAttention`
- **DirectML:** ✓ compatible (~12× speedup over CPU in eval)

In [ ]:
args_drspaam = _default_args(
    detector    = 'drspaam',
    dataset     = 'frog',
    epochs      = 30,
    lr          = 1e-3,
    dropout     = 0.5,
    patience    = 5,
    lr_schedule = 'plateau',
    auc_every   = 5,
    out         = _nb_dir / 'weights_drspaam_frog.pth',
)

history_drspaam = train_model(args_drspaam)
plot_history(history_drspaam, 'DrSpaamDetector')

## Model 3 — FullScanCNNDetector

Custom full-scan model using a **dilated 1-D CNN** over all beams followed by a **GRU** for temporal aggregation.

- **Input:** aligned full scan `(N_beams, T=5, 3)` — range, x, y
- **Architecture:** DilatedScanBackbone → GRU over time → per-beam head
- **GPU (CUDA/ROCm):** ✓ fully supported

> **Local Windows / DirectML:** add `force_cpu=True` to the args below.

Tunable: `backbone_channels`, `hidden`, `dropout`

In [ ]:
args_fscnn = _default_args(
    detector          = 'fullscan_cnn',
    dataset           = 'frog',
    epochs            = 30,
    lr                = 1e-3,
    dropout           = 0.3,
    backbone_channels = 64,
    hidden            = 128,
    patience          = 5,
    lr_schedule       = 'plateau',
    auc_every         = 5,
    out               = _nb_dir / 'weights_fullscan_cnn_frog.pth',
    # force_cpu = True,   # ← uncomment if using DirectML locally
)

history_fscnn = train_model(args_fscnn)
plot_history(history_fscnn, 'FullScanCNNDetector')

## Model 4 — SpaceTimeCNNDetector

Custom model treating the scan history as a **2-D image** `(N_beams × T)` and applying
standard 2-D convolutions across both spatial and temporal axes simultaneously.

- **Input:** aligned full scan `(N_beams, T=5, 3)` reshaped to `(3, T, N_beams)`
- **Architecture:** two 2-D conv layers with (beam × time) kernels → per-beam head
- **GPU (CUDA/ROCm):** ✓ fully supported

Tunable: `out_channels`, `dropout`

In [ ]:
args_stcnn = _default_args(
    detector     = 'spacetime_cnn',
    dataset      = 'frog',
    epochs       = 30,
    lr           = 1e-3,
    dropout      = 0.2,
    out_channels = 128,
    patience     = 5,
    lr_schedule  = 'plateau',
    auc_every    = 5,
    out          = _nb_dir / 'weights_spacetime_cnn_frog.pth',
)

history_stcnn = train_model(args_stcnn)
plot_history(history_stcnn, 'SpaceTimeCNNDetector')

## Model 5 — FullScanTransformerDetector

Most expressive custom model: **dilated CNN** spatial encoder + **global beam self-attention** + **GRU** temporal aggregation.

- **Input:** aligned full scan `(N_beams, T=5, 3)`
- **Architecture:** DilatedScanBackbone → multi-head self-attention over beams → GRU over time → per-beam head
- **GPU (CUDA/ROCm):** ✓ fully supported

> **Local Windows / DirectML:** add `force_cpu=True` to the args below.

Tunable: `backbone_channels`, `n_heads`, `hidden`, `dropout`

In [ ]:
args_fstransformer = _default_args(
    detector          = 'fullscan_transformer',
    dataset           = 'frog',
    epochs            = 30,
    lr                = 5e-4,
    dropout           = 0.2,
    backbone_channels = 64,
    n_heads           = 8,
    hidden            = 128,
    patience          = 5,
    lr_schedule       = 'plateau',
    auc_every         = 5,
    out               = _nb_dir / 'weights_fullscan_transformer_frog.pth',
    # force_cpu = True,   # ← uncomment if using DirectML locally
)

history_fstransformer = train_model(args_fstransformer)
plot_history(history_fstransformer, 'FullScanTransformerDetector')

## Final Comparison — Test Set

Each trained checkpoint is loaded and evaluated on the **held-out test split**.

- **Neural models** — agnostic / per-class AUC (area under the precision-recall curve)  
- **AlgorithmicDetector** — F1 at 0.5 m matching radius (no confidence score → no PR curve)

The grouped bar chart shows all four metrics side by side.  
For the AlgorithmicDetector bar only F1 is shown (agnostic column); class-specific bars are omitted.

In [ ]:
_MODELS = {
    'DROW':              ('drow',                 _nb_dir / 'weights_drow_frog.pth'),
    'DR-SPAAM':          ('drspaam',              _nb_dir / 'weights_drspaam_frog.pth'),
    'FullScanCNN':       ('fullscan_cnn',         _nb_dir / 'weights_fullscan_cnn_frog.pth'),
    'SpaceTimeCNN':      ('spacetime_cnn',        _nb_dir / 'weights_spacetime_cnn_frog.pth'),
    'FullScanTransform': ('fullscan_transformer', _nb_dir / 'weights_fullscan_transformer_frog.pth'),
}

# Evaluate all neural models on the test split
neural_results = {}
for label, (det, ckpt) in _MODELS.items():
    print(f'Evaluating {label} …')
    aucs = eval_neural(det, ckpt, split='test')
    neural_results[label] = aucs
    if aucs:
        print(f'  agnostic={aucs["agnostic"]:.1%}  wc={aucs["wc"]:.1%}'
              f'  wa={aucs["wa"]:.1%}  wp={aucs["wp"]:.1%}')

# Print summary table
print()
print(f"{'Model':<20}  {'Agnostic':>9}  {'WC':>7}  {'WA':>7}  {'WP':>7}")
print('-' * 58)
for label, aucs in neural_results.items():
    if aucs:
        print(f"{label:<20}  {aucs['agnostic']:>9.1%}  {aucs['wc']:>7.1%}"
              f"  {aucs['wa']:>7.1%}  {aucs['wp']:>7.1%}")
    else:
        print(f"{label:<20}  {'FAILED':>9}")

print(f"{'Algorithmic':>20}  {'—':>9}  {'—':>7}  {'—':>7}  {'—':>7}"
      f"  F1={algo_result['f1']:.1%}")

# Bar chart
plot_comparison(neural_results, algo_result)